In [43]:
import pandas as pd

In [44]:
# Load Olist orders dataset
orders = pd.read_csv("olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
])

In [45]:
order_items = pd.read_csv("olist_order_items_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
translation = pd.read_csv("product_category_name_translation.csv")

In [54]:
cancelled = orders[orders["order_approved_at"].isna()]
cancel_rate = len(cancelled) / len(orders)
print(cancel_rate)

0.0016089942780140988


In [46]:
# Join order_items -> products -> translation
items_with_english = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
).merge(
    translation,
    on="product_category_name",
    how="left"
)

In [47]:
# --- 1. Inter-arrival time distribution ---
orders_sorted = orders.sort_values("order_purchase_timestamp")
purchase_times = orders_sorted["order_purchase_timestamp"].dropna()
inter_arrival = purchase_times.diff().dropna().dt.total_seconds()
inter_arrival_samples = inter_arrival.quantile([0.25, 0.5, 0.75]).tolist()
print("Inter-arrival time (seconds):")
print(inter_arrival.describe())

Inter-arrival time (seconds):
count    9.944000e+04
mean     6.714974e+02
std      1.917385e+04
min      0.000000e+00
25%      8.300000e+01
50%      2.220000e+02
75%      5.070000e+02
max      5.410280e+06
Name: order_purchase_timestamp, dtype: float64


In [48]:
# --- 2. Approval delay ---
approval_delay = (
    orders["order_approved_at"] - orders["order_purchase_timestamp"]
).dt.total_seconds().dropna()
approval_samples = approval_delay.quantile([0.25, 0.5, 0.75]).tolist()
print("\nApproval delay (seconds):")
print(approval_delay.describe())


Approval delay (seconds):
count    9.928100e+04
mean     3.750874e+04
std      9.373681e+04
min      0.000000e+00
25%      7.740000e+02
50%      1.236000e+03
75%      5.249100e+04
max      1.623305e+07
dtype: float64


In [49]:
# --- 3. Dispatch delay ---
dispatch_delay = (
    orders["order_delivered_carrier_date"] - orders["order_approved_at"]
).dt.total_seconds().dropna()
dispatch_samples = dispatch_delay.quantile([0.25, 0.5, 0.75]).tolist()
print("\nDispatch delay (seconds):")
print(dispatch_delay.describe())


Dispatch delay (seconds):
count    9.764400e+04
mean     2.423553e+05
std      3.066705e+05
min     -1.479332e+07
25%      7.564400e+04
50%      1.571095e+05
75%      3.093525e+05
max      1.086589e+07
dtype: float64


In [50]:
# --- 4. Delivery delay ---
delivery_delay = (
    orders["order_delivered_customer_date"] - orders["order_delivered_carrier_date"]
).dt.total_seconds().dropna()
delivery_samples = delivery_delay.quantile([0.25, 0.5, 0.75]).tolist()
print("\nDelivery delay (seconds):")
print(delivery_delay.describe())


Delivery delay (seconds):
count    9.647500e+04
mean     8.061593e+05
std      7.568745e+05
min     -1.390709e+06
25%      3.542355e+05
50%      6.134200e+05
75%      1.039316e+06
max      1.772850e+07
dtype: float64


In [51]:
# --- 5. Category skew (from items dataset) ---
category_counts = items_with_english["product_category_name_english"].value_counts()
top_categories = category_counts.head(10).index.tolist()
print("Top categories:", top_categories)

Top categories: ['bed_bath_table', 'health_beauty', 'sports_leisure', 'furniture_decor', 'computers_accessories', 'housewares', 'watches_gifts', 'telephony', 'garden_tools', 'auto']


In [52]:
# --- 6. Order value distribution (from payments dataset) ---
order_values = payments.groupby("order_id")["payment_value"].sum()
order_value_samples = order_values.quantile([0.25, 0.5, 0.75]).tolist()
print("\nOrder value distribution:")
print(order_values.describe())


Order value distribution:
count    99440.000000
mean       160.990267
std        221.951257
min          0.000000
25%         62.010000
50%        105.290000
75%        176.970000
max      13664.080000
Name: payment_value, dtype: float64


In [53]:
# --- Final output: numbers to hardcode into Java workload generator ---
print("\n=== VALUES TO COPY INTO JAVA WORKLOAD GENERATOR ===")
print("inter_arrival_samples =", inter_arrival_samples)
print("approval_samples =", approval_samples)
print("dispatch_samples =", dispatch_samples)
print("delivery_samples =", delivery_samples)
print("order_value_samples =", order_value_samples)
print("top_categories =", category_counts.head(10).index.tolist())


=== VALUES TO COPY INTO JAVA WORKLOAD GENERATOR ===
inter_arrival_samples = [83.0, 222.0, 507.0]
approval_samples = [774.0, 1236.0, 52491.0]
dispatch_samples = [75644.0, 157109.5, 309352.5]
delivery_samples = [354235.5, 613420.0, 1039315.5]
order_value_samples = [62.01, 105.29, 176.97]
top_categories = ['bed_bath_table', 'health_beauty', 'sports_leisure', 'furniture_decor', 'computers_accessories', 'housewares', 'watches_gifts', 'telephony', 'garden_tools', 'auto']
